In [13]:
from load_data import (
    load_bodies_from_csv
)
from url_helpers import (
    build_email_url_map,
    build_email_url_artifact_map
)

#CSV data Paths
data_dir = "test_files"
csv_path = f"{data_dir}/TREC-07-only-phishing-6m.csv"

bodies = load_bodies_from_csv(csv_path)
url_map = build_email_url_map(bodies, normalize=True)
print(url_map[1])




['http://ctmay.com']


In [14]:
# 1) Extract URLs per email (normalized)
url_map = build_email_url_map(bodies, normalize=True)

# 2) Convert URLs into artifacts per email
artifact_map = build_email_url_artifact_map(url_map)

In [15]:
import torch 
from ml_url_src import URLEncoder, tokenize

bodies = load_bodies_from_csv(csv_path)
url_map = build_email_url_map(bodies, normalize=True)
def flatten_url_map(url_map):
    """
    url_map: Dict[int, List[str] | None]
    Returns: List[str] of URLs, flattened, stripped, non-empty, non-None.
    """
    urls = []
    for _, lst in url_map.items():
        if not lst:
            continue
        for u in lst:
            if u is None:
                continue
            if not isinstance(u, str):
                continue
            u = u.strip()
            if u:
                urls.append(u)
    return urls

urls = flatten_url_map(url_map)
print("Flattened URL count:", len(urls))
print("Example:", urls[0])

def load_model(path, model_class, device="cpu"):
    checkpoint = torch.load(path, map_location="cpu")

    raw_config = checkpoint.get("config", {})

    # ✅ keep only args that URLEncoder.__init__ accepts
    allowed = {
        "embed_dim",
        "hidden_size",
        "num_hidden_layers",
        "num_attention_heads",
        "intermediate_size",
        "max_len",
    }
    model_config = {k: v for k, v in raw_config.items() if k in allowed}

    model = model_class(**model_config)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)

    optimizer = None
    opt_state = checkpoint.get("optimizer_state_dict", None)
    if opt_state is not None:
        optimizer = torch.optim.AdamW(model.parameters())
        optimizer.load_state_dict(opt_state)

    print(f"Loaded model from epoch {checkpoint.get('epoch', '?')}")
    return model, optimizer, checkpoint.get("epoch", None), raw_config

model, _, _, config = load_model("best_model_moco.pt", URLEncoder, device="cpu")
model.eval()



Flattened URL count: 28494
Example: http://www.moujsjkhchum.com


/var/folders/y6/40jhvb_s5jd4t0z5v4h7ms7c0000gn/T/ipykernel_96756/1267229696.py:30: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(path, map_location="

Loaded model from epoch 9


URLEncoder(
  (encoder): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 384, padding_idx=1)
      (token_type_embeddings): Embedding(2, 384)
      (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(130, 384, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (LayerNorm): LayerNorm(

In [16]:
import numpy as np
import torch
from tqdm import tqdm

@torch.no_grad()
def embed_urls(model, urls, batch_size=512, device="cpu"):
    model.eval()
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    model.to(device)

    all_emb = []

    for i in tqdm(range(0, len(urls), batch_size), desc="Embedding"):
        batch = urls[i:i+batch_size]

        inputs = tokenize(batch)
        input_ids = inputs["input_ids"].to(device).long()
        attn_mask = inputs["attention_mask"].to(device)

        z = model(input_ids, attn_mask)          # (B, D), already normalized in your model
        all_emb.append(z.detach().cpu().numpy())

    return np.vstack(all_emb)

embeddings = embed_urls(model, urls, batch_size=512)
print(embeddings.shape)



Embedding: 100%|██████████| 56/56 [00:51<00:00,  1.09it/s]

(28494, 256)


In [7]:
import hdbscan

def run_hdbscan(embeddings, min_cluster_size=5, min_samples=5):
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method="eom",
    )
    labels = clusterer.fit_predict(embeddings)
    return labels, clusterer

labels, clusterer = run_hdbscan(embeddings, min_cluster_size=5, min_samples=5)
#print("num clusters (excluding noise):", len(set(labels)) - (1 if -1 in labels else 0))
#print("noise points:", np.sum(labels == -1))

from collections import Counter, defaultdict

def cluster_report(urls, labels, top_n=20):
    counts = Counter(labels)
    # Remove noise (-1) from ranking
    cluster_sizes = [(cid, sz) for cid, sz in counts.items() if cid != -1]
    cluster_sizes.sort(key=lambda x: x[1], reverse=True)

    print(f"Total URLs: {len(urls)}")
    print(f"Noise (-1): {counts.get(-1, 0)}")
    print(f"Clusters: {len(cluster_sizes)}")
    print("\nTop clusters:")
    for rank, (cid, sz) in enumerate(cluster_sizes[:top_n], start=1):
        print(f"{rank:>2}. cluster_id={cid:<5} size={sz}")
    return cluster_sizes

def urls_in_cluster(urls, labels, cluster_id, max_show=200):
    idxs = np.where(labels == cluster_id)[0]
    print(f"cluster_id={cluster_id} size={len(idxs)}")
    for i in idxs[:max_show]:
        print(urls[i])
    if len(idxs) > max_show:
        print(f"... ({len(idxs) - max_show} more)")


def urls_in_cluster_ranked(urls, labels, clusterer, cluster_id, max_show=200):
    probs = getattr(clusterer, "probabilities_", None)
    if probs is None:
        return urls_in_cluster(urls, labels, cluster_id, max_show=max_show)

    idxs = np.where(labels == cluster_id)[0]
    idxs_sorted = sorted(idxs, key=lambda i: probs[i], reverse=True)

    print(f"cluster_id={cluster_id} size={len(idxs)} (ranked by HDBSCAN probability)")
    for i in idxs_sorted[:max_show]:
        print(f"{probs[i]:.3f}  {urls[i]}")
    if len(idxs_sorted) > max_show:
        print(f"... ({len(idxs_sorted) - max_show} more)")

import numpy as np

def show_noise_points(urls, labels, max_show=200, descending=True):
    """
    Prints HDBSCAN noise URLs (label == -1), sorted alphabetically.
    """
    labels = np.asarray(labels)
    noise_idxs = np.where(labels == -1)[0]
    noise_urls = [urls[i] for i in noise_idxs]

    # Sort alphabetically (case-insensitive), descending by default
    noise_urls_sorted = sorted(noise_urls, key=lambda s: s.lower(), reverse=descending)

    print(f"Noise points: {len(noise_urls_sorted)} / {len(urls)} ({len(noise_urls_sorted)/len(urls)*100:.2f}%)")
    for u in noise_urls_sorted[:max_show]:
        print(u)

    if len(noise_urls_sorted) > max_show:
        print(f"... ({len(noise_urls_sorted) - max_show} more)")

    return noise_urls_sorted

/Users/mcandersyo/.venv-pyg/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/mcandersyo/.venv-pyg/lib/python3.12/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


KeyboardInterrupt: 

In [1]:
#cluster_sizes = cluster_report(urls, labels, top_n=30)

# Inspect largest cluster
#cluster_sizes = cluster_report(urls, labels, top_n=5)
#largest_cluster_id = cluster_sizes[1][0]
#urls_in_cluster(urls, labels, largest_cluster_id, max_show=20)
#urls_in_cluster_ranked(urls, labels, clusterer, largest_cluster_id, max_show=50)
# usage:
noise_idxs = show_noise_points(urls, labels, max_show=200)



NameError: name 'show_noise_points' is not defined

In [25]:
import numpy as np

def knn_indices(embeddings, query_index, k=20):
    """
    Returns (indices, similarities) of the top-k nearest neighbors for a given index.
    embeddings: (N, D) numpy array, assumed L2-normalized.
    """
    q = embeddings[query_index]  # (D,)
    sims = embeddings @ q        # (N,)
    # get top k+1 to include itself, then remove itself
    top = np.argpartition(-sims, k+1)[:k+1]
    top = top[np.argsort(-sims[top])]

    # remove self if present
    top = [i for i in top if i != query_index][:k]
    return top, sims[top]

def show_knn(urls, embeddings, query, k=20):
    """
    query can be:
      - an int index into urls
      - a URL string (exact match)
    """
    if isinstance(query, int):
        qi = query
    else:
        # find first exact match
        try:
            qi = urls.index(query)
        except ValueError:
            raise ValueError("Query URL not found in urls list (exact match). Try index-based query.")

    nbrs, sims = knn_indices(embeddings, qi, k=k)

    print("QUERY:")
    print(f"[{qi}] {urls[qi]}")

    print(f"Top {k} neighbors:")
    for rank, (idx, sim) in enumerate(zip(nbrs, sims), start=1):
        print(f"{rank:>2}. sim={sim:.4f}  [{idx}] {urls[idx]}")

show_knn(urls, embeddings, 4500, k=100)


QUERY:
[4500] http://via.flshey.com/?58836E1FA4C6FDC25A4151EF8832437949976019ACD5F2C6751C42EB&t4
Top 100 neighbors:
 1. sim=1.0000  [4498] http://via.flshey.com/?58836E1FA4C6FDC25A4151EF8832437949976019ACD5F2C6751C42EB&t0
 2. sim=0.9993  [4487] http://via.flshey.com/?5F81750EA1C6F2C55A4151EF8832437949976019ACD5F2C6751C42EB&t4
 3. sim=0.9992  [4485] http://via.flshey.com/?5F81750EA1C6F2C55A4151EF8832437949976019ACD5F2C6751C42EB&t0
 4. sim=0.9992  [4469] http://via.flshey.com/?5D836C5DF9E7EEC57D000FE78C22527949976019ACD5F2C6751C42EB&t4
 5. sim=0.9992  [4467] http://via.flshey.com/?5D836C5DF9E7EEC57D000FE78C22527949976019ACD5F2C6751C42EB&t0
 6. sim=0.9983  [4461] http://via.flshey.com/?5D836C5DF9E7EEC57D1C54FD8C225F25508F6E43AAC6&t0
 7. sim=0.9983  [4463] http://via.flshey.com/?5D836C5DF9E7EEC57D1C54FD8C225F25508F6E43AAC6&t4
 8. sim=0.9982  [4517] http://via.flshey.com/?5B966202BBCAFFCA5A4151EF8832437949976019ACD5F2C6751C42EB&t0
 9. sim=0.9982  [4650] http://via.flshey.com/?5B966202BBCAFF

In [26]:
import numpy as np

from text_embeddings import (
    train_char_tfidf_model,
    train_tfidf_svd_reducer,
    get_char_embeddings,
)


def fit_tfidf_url_embeddings(
    urls,
    *,
    analyzer: str = "char_wb",
    ngram_range=(3, 5),
    min_df: int = 2,
    max_features: int = 200_000,
    out_dim: int = 256,
    seed: int = 42,
    l2_normalize: bool = True,
):
    tfidf_model, X_train = train_char_tfidf_model(
        urls,
        analyzer=analyzer,
        ngram_range=ngram_range,
        min_df=min_df,
        max_features=max_features,
    )
    svd_model = train_tfidf_svd_reducer(X_train, out_dim=out_dim, seed=seed)
    embeddings = get_char_embeddings(
        urls,
        tfidf_model=tfidf_model,
        svd_reducer=svd_model,
        l2_normalize=l2_normalize,
    )
    embeddings = np.asarray(embeddings, dtype=np.float32)
    return tfidf_model, svd_model, embeddings


def knn_indices(embeddings, query_index, k=20):
    """
    Returns (indices, similarities) of the top-k nearest neighbors for a given index.
    embeddings: (N, D) numpy array, assumed L2-normalized.
    """
    q = embeddings[query_index]  # (D,)
    sims = embeddings @ q        # (N,)
    # get top k+1 to include itself, then remove itself
    top = np.argpartition(-sims, k + 1)[: k + 1]
    top = top[np.argsort(-sims[top])]

    # remove self if present
    top = [i for i in top if i != query_index][:k]
    return top, sims[top]


def show_knn(urls, embeddings, query, k=20):
    """
    query can be:
      - an int index into urls
      - a URL string (exact match)
    """
    if isinstance(query, int):
        qi = query
    else:
        # find first exact match
        try:
            qi = urls.index(query)
        except ValueError:
            raise ValueError(
                "Query URL not found in urls list (exact match). Try index-based query."
            )

    nbrs, sims = knn_indices(embeddings, qi, k=k)

    print("QUERY:")
    print(f"[{qi}] {urls[qi]}")

    print(f"Top {k} neighbors:")
    for rank, (idx, sim) in enumerate(zip(nbrs, sims), start=1):
        print(f"{rank:>2}. sim={sim:.4f}  [{idx}] {urls[idx]}")


# Example usage
# Assumes you already built `urls: List[str]`
#tfidf_model, svd_model, tfidf_emb = fit_tfidf_url_embeddings(urls)
show_knn(urls, tfidf_emb, 4500, k=100)


QUERY:
[4500] http://via.flshey.com/?58836E1FA4C6FDC25A4151EF8832437949976019ACD5F2C6751C42EB&t4
Top 100 neighbors:
 1. sim=0.9998  [4498] http://via.flshey.com/?58836E1FA4C6FDC25A4151EF8832437949976019ACD5F2C6751C42EB&t0
 2. sim=0.9888  [4962] http://iat.flshey.com/?58836E1FA4C6FDC25A4151EF8832437949976019ACD5F2C6751C42EB&t4
 3. sim=0.9881  [4960] http://iat.flshey.com/?58836E1FA4C6FDC25A4151EF8832437949976019ACD5F2C6751C42EB&t0
 4. sim=0.9851  [4267] http://evi.flshey.com/?58836E1FA4C6FDC25A4151EF8832437949976019ACD5F2C6751C42EB&t4
 5. sim=0.9843  [4265] http://evi.flshey.com/?58836E1FA4C6FDC25A4151EF8832437949976019ACD5F2C6751C42EB&t0
 6. sim=0.9834  [2681] http://flshey.com/?58836E1FA4C6FDC25A4151EF8832437949976019ACD5F2C6751C42EB&t4
 7. sim=0.9828  [2679] http://flshey.com/?58836E1FA4C6FDC25A4151EF8832437949976019ACD5F2C6751C42EB&t0
 8. sim=0.9824  [5447] http://tur.flshey.com/?58836E1FA4C6FDC25A4151EF8832437949976019ACD5F2C6751C42EB&t4
 9. sim=0.9817  [5445] http://tur.flshey.com